# 04 — Replication in an alcohol use disorder study (Wyant et al., 2024)

Logistic regression, random forest, XGBoost, GRU and GRU-D with the Smoke Free settings (no re-tuning), original and expanded
features, both tasks. Design: participant-level 10-fold CV (KFold, seed 42); the authors' analytic sample of 151 participants. Pooled AUC over all out-of-fold predictions with 2,000 participant-level bootstrap
resamples (percentile CIs, the same resamples for every configuration), plus per-participant AUC.

**Produces:** Table 9 (EMASENS/alcohol columns), Figure 5 and Figure 8 values. Data: see `data/README.md`.

In [ ]:
# ════════════════════════════════════════════════════════
# CELL 1 — SETUP
# ════════════════════════════════════════════════════════
import os, json, time, random, copy, subprocess, sys
import numpy as np, pandas as pd
try:
    import pyreadr
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'pyreadr', '--quiet']); import pyreadr
try:
    import xgboost as xgb
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'xgboost', '--quiet']); import xgboost as xgb
import torch, torch.nn as nn
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from scipy import stats
import warnings; warnings.filterwarnings('ignore')

EMA_PATH  = 'ema.csv'                         # Colab: '/content/ema.csv'
IDS_PATH  = 'baseline_features_1week_v3.csv'  # Colab: '/content/baseline_features_1week_v3.csv'
for _v in ('EMA_PATH', 'IDS_PATH'):                # fall back to the repository data/ folder
    if not os.path.exists(globals()[_v]) and os.path.exists('../data/' + globals()[_v]): globals()[_v] = '../data/' + globals()[_v]
SAVE_DIR  = 'alcohol_replication_results'  # Colab + Drive: '/content/drive/MyDrive/alcohol_replication_results'
SEED      = 42
QUICK_TEST = False   # True = 20 participants, 1 epoch (checks the code runs)
DEEP_EPOCHS = 1 if QUICK_TEST else 30
DEEP_BS    = 16
N_FOLDS    = 10
N_BOOT     = 200 if QUICK_TEST else 2000
if QUICK_TEST: SAVE_DIR += '_quicktest'
os.makedirs(os.path.join(SAVE_DIR, 'predictions'), exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"device={device}  QUICK_TEST={QUICK_TEST}  results -> {SAVE_DIR}")


In [ ]:
# ════════════════════════════════════════════════════════
# CELL 2 — LOAD, RESTRICT TO THE ANALYSIS SAMPLE, BUILD FEATURES
# Engineered features use the same definitions as the Smoke Free notebook.
# ════════════════════════════════════════════════════════
for p in (EMA_PATH, IDS_PATH):
    if not os.path.exists(p):
        raise FileNotFoundError(f"{p} not found. Upload it and set the path in Cell 1.")
raw = pd.read_csv(EMA_PATH, low_memory=False)
ids = pd.read_csv(IDS_PATH, usecols=['subid', 'dttm_label'])
ids['dttm_label'] = pd.to_datetime(ids['dttm_label'], utc=True)
period = ids.groupby('subid')['dttm_label'].agg(['min', 'max'])
print(f"ema.csv: {len(raw)} EMAs, {raw['subid'].nunique()} participants; analysis sample: {len(period)} participants")

raw['t'] = pd.to_datetime(raw['dttm_obs'], utc=True)
df = raw[raw['subid'].isin(period.index)].join(period, on='subid')
# study period: from 1 day before the first prediction window to 1 week after the last (window = next week)
df = df[(df['t'] >= df['min'] - pd.Timedelta(days=1)) & (df['t'] <= df['max'] + pd.Timedelta(days=7))]
df['id'] = df['subid'].astype(str)
df = df.sort_values(['id', 't']).reset_index(drop=True)
if QUICK_TEST:
    df = df[df['id'].isin(sorted(df['id'].unique())[:20])].reset_index(drop=True)
df['label'] = (df['ema_1'] == 'Yes').astype(float)
df['EMA_date_time'] = df['t'].dt.tz_convert(None)          # naive UTC, used by Cells 3-4
local = df['t'].dt.tz_convert('America/Chicago')           # study site time zone, for clock features
print(f"Kept {len(df)} EMAs, {df['id'].nunique()} participants, lapse rate {df['label'].mean():.2%}")

ITEMS = {'ema_2': 'craving', 'ema_3': 'risky_situations', 'ema_4': 'stressful_events',
         'ema_5': 'pleasant_events', 'ema_6': 'valence', 'ema_7': 'arousal'}
MORNING = {'ema_8': 'anticipated_risk', 'ema_9': 'anticipated_stress', 'ema_10': 'abstinence_efficacy'}
df = df.rename(columns={**ITEMS, **MORNING})
g = df.groupby('id', sort=False)
for c in ITEMS.values():                                   # 1.5% missing: carry the person's last answer
    df[c] = g[c].ffill()
# morning items: carry the day's morning answers to later EMAs on the same local day
df['local_day'] = local.dt.date
df['is_morning'] = (df['ema_type'] == 'morning').astype(float)
df['morning_carried'] = 0.0
for c in MORNING.values():
    filled = df.groupby(['id', 'local_day'], sort=False)[c].ffill()
    df.loc[df[c].isna() & filled.notna(), 'morning_carried'] = 1.0
    df[c] = filled
df['morning_missing'] = df[list(MORNING.values())].isna().any(axis=1).astype(float)
for c in list(ITEMS.values()) + list(MORNING.values()):
    df[c] = df[c].fillna(df[c].median())                     # remaining gaps (start of a day / person)

EPOCH = pd.Timestamp('1970-01-01')
g = df.groupby('id', sort=False)
df['prior_event_lapse'] = g['label'].shift(1).fillna(0)
df['mins_since_prev_rep'] = g['EMA_date_time'].diff().dt.total_seconds().div(60).fillna(0).clip(lower=0)
start = g['EMA_date_time'].transform('min')
df['days_since_qd'] = (df['EMA_date_time'] - start).dt.total_seconds() / 86400   # days since study start

# ── 13 engineered features (severity = craving) ────────
hour = local.dt.hour.astype(float)
df['hour_sin'] = np.sin(2 * np.pi * hour / 24); df['hour_cos'] = np.cos(2 * np.pi * hour / 24)
df['is_weekend'] = local.dt.dayofweek.ge(5).astype(float).to_numpy()
g = df.groupby('id', sort=False)
df['severity_change'] = g['craving'].diff().fillna(0)
df['rolling3_sev_mean'] = g['craving'].transform(lambda v: v.shift(1).rolling(3, min_periods=1).mean()).fillna(0)
df['rolling3_sev_std'] = g['craving'].transform(lambda v: v.shift(1).rolling(3, min_periods=1).std()).fillna(0)
ts = (df['EMA_date_time'] - EPOCH).dt.total_seconds()
df['_lt'] = np.where(df['label'].eq(1), ts, np.nan)
last_lapse = df.groupby('id', sort=False)['_lt'].transform(lambda v: v.shift(1).ffill())
df['days_since_last_lapse'] = ((ts - last_lapse.fillna((start - EPOCH).dt.total_seconds())) / 86400).clip(lower=0)
df['log_days_since_qd'] = np.log1p(df['days_since_qd'].clip(lower=0))
g = df.groupby('id', sort=False)
for k in (1, 2, 3): df[f'sev_lag{k}'] = g['craving'].shift(k).fillna(0)
for k in (2, 3):    df[f'lapse_lag{k}'] = g['label'].shift(k).fillna(0)
df = df.drop(columns=['_lt'])
EXTRA = ['hour_sin', 'hour_cos', 'is_weekend', 'severity_change', 'rolling3_sev_mean',
         'rolling3_sev_std', 'days_since_last_lapse', 'log_days_since_qd',
         'sev_lag1', 'sev_lag2', 'sev_lag3', 'lapse_lag2', 'lapse_lag3']

# person-level baseline (check 4): running lapse rate before this EMA, no model
df['baseline_score'] = g['label'].transform(lambda v: v.shift(1).expanding().mean()).fillna(0)

NATIVE_ORIG = (list(ITEMS.values()) + list(MORNING.values()) +
               ['is_morning', 'morning_carried', 'morning_missing',
                'days_since_qd', 'mins_since_prev_rep', 'prior_event_lapse'])
FEATURE_SETS = {'native_orig': NATIVE_ORIG, 'native_exp': NATIVE_ORIG + EXTRA}
allf = sorted(set(sum(FEATURE_SETS.values(), [])))
df[allf] = df[allf].astype(float)
assert not df[allf].isna().any().any(), "missing feature values"
for k, v in FEATURE_SETS.items(): print(f"{k:12s}: {len(v)} features")

n_per = df.groupby('id').size(); lap = df.groupby('id')['label'].sum()
gap = df.groupby('id')['EMA_date_time'].diff().dt.total_seconds().div(60).dropna()
print(f"\nEMAs per participant: median {n_per.median():.0f} (range {n_per.min()}-{n_per.max()})")
print(f"Participants with >=1 lapse: {(lap>0).sum()}; with >=3 lapses and >=3 non-lapses: {((lap>=3)&(n_per-lap>=3)).sum()}")
print(f"Gap between EMAs: median {gap.median():.0f} min, IQR {gap.quantile(.25):.0f}-{gap.quantile(.75):.0f}, "
      f"SD of log(1+gap) {np.log1p(gap).std():.2f}  (EMASENS 0.68; Smoke Free 2.23)")


In [ ]:
# ════════════════════════════════════════════════════════
# CELL 3 — MODELS (same architectures and fixed settings as the main study)
# ════════════════════════════════════════════════════════
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

def fit_flat(mtype, X, y):
    sc = StandardScaler().fit(X); Xs = sc.transform(X); y = y.astype(int)
    pos = max((y == 1).sum(), 1); neg = (y == 0).sum()
    if mtype == 'LR':
        clf = LogisticRegression(C=1.0, max_iter=1000, class_weight='balanced', random_state=42)
    elif mtype == 'RF':
        clf = RandomForestClassifier(n_estimators=500, min_samples_leaf=5, class_weight='balanced',
                                     n_jobs=-1, random_state=42)
    elif mtype == 'XGB':
        clf = xgb.XGBClassifier(n_estimators=500, max_depth=6, learning_rate=0.1, subsample=0.8,
                                colsample_bytree=0.8, scale_pos_weight=neg / pos,
                                eval_metric='logloss', n_jobs=-1, random_state=42, verbosity=0)
    clf.fit(Xs, y)
    return lambda Xn: clf.predict_proba(sc.transform(Xn))[:, 1]

class PredHead(nn.Module):
    def __init__(self, hid=64, drop=0.3):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(hid, hid // 2), nn.GELU(), nn.Dropout(drop), nn.Linear(hid // 2, 1))
    def forward(self, h): return self.net(h).squeeze(-1)

class GRUEncoder(nn.Module):
    def __init__(self, nf, hid=64, drop=0.3):
        super().__init__(); self.gru = nn.GRU(nf, hid, 2, batch_first=True, dropout=drop)
    def forward(self, x, lengths=None):
        if lengths is None: return self.gru(x)[0]
        pk = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        return pad_packed_sequence(self.gru(pk)[0], batch_first=True, total_length=x.size(1))[0]

class GRUDCell(nn.Module):
    """Che et al. (2018): hidden-state and input decay from elapsed time."""
    def __init__(self, inp, hid):
        super().__init__()
        gi = inp * 2 + hid
        self.Wr, self.Wz, self.Wh = nn.Linear(gi, hid), nn.Linear(gi, hid), nn.Linear(gi, hid)
        self.Wgh, self.Wgx = nn.Linear(1, hid), nn.Linear(inp, inp)
        self.h0 = nn.Parameter(torch.zeros(hid)); self.register_buffer('xm', torch.zeros(inp))
    def forward(self, x, m, d_feat, d_gap, h):
        gh = torch.exp(-torch.clamp(self.Wgh(d_gap), min=0))
        ht = gh * h + (1 - gh) * self.h0.unsqueeze(0).expand_as(h)
        gx = torch.exp(-torch.clamp(self.Wgx(d_feat), min=0))
        xt = m * x + (1 - m) * (gx * x + (1 - gx) * self.xm)
        inp = torch.cat([xt, m, ht], 1)
        r, z = torch.sigmoid(self.Wr(inp)), torch.sigmoid(self.Wz(inp))
        return (1 - z) * ht + z * torch.tanh(self.Wh(torch.cat([xt, m, r * ht], 1)))

class GRUDEncoder(nn.Module):
    """Input layout: [features | mask | per-feature delta]; GRU-D cell + 1 GRU layer."""
    def __init__(self, nf, hid=64, drop=0.3):
        super().__init__(); self.nf = nf
        self.cell = GRUDCell(nf, hid); self.gru2 = nn.GRU(hid, hid, 1, batch_first=True)
        self.drop = nn.Dropout(drop)
    def forward(self, x, lengths=None):
        B, T, _ = x.shape; nf = self.nf
        f, m, d = x[:, :, :nf], x[:, :, nf:2 * nf], x[:, :, 2 * nf:]
        h = self.cell.h0.unsqueeze(0).expand(B, -1).contiguous(); outs = []
        for t in range(T):
            h = self.cell(f[:, t], m[:, t], d[:, t], d[:, t].mean(-1, keepdim=True), h); outs.append(h)
        out, _ = self.gru2(self.drop(torch.stack(outs, 1)))
        return self.drop(out)

def collate(batch):
    xs, ys = zip(*batch); L = torch.tensor([len(x) for x in xs]); T = int(L.max())
    X = torch.zeros(len(xs), T, xs[0].shape[1]); Y = torch.full((len(xs), T), float('nan'))
    for i, (x, y) in enumerate(zip(xs, ys)):
        X[i, :len(x)] = torch.from_numpy(x); Y[i, :len(x)] = torch.from_numpy(y)
    return X, Y, L

def build_seqs(feats, grud):
    """One (X, y, id) per participant; GRU-D input adds mask (all 1s) and time gap."""
    out = []
    for pid, gdf in df.groupby('id', sort=True):
        X = gdf[feats].to_numpy(np.float32); y = gdf['label'].to_numpy(np.float32)
        if grud:
            gaps = gdf['EMA_date_time'].diff().dt.total_seconds().div(60).fillna(0).clip(lower=0).to_numpy()
            delta = (np.log1p(gaps) / np.log1p(60 * 24 * 7)).astype(np.float32)
            X = np.concatenate([X, np.ones_like(X), np.repeat(delta[:, None], X.shape[1], 1)], 1)
        out.append((X, y, pid))
    return out

def train_predict_deep(mtype, train, test, nf, task):
    """Standardise features on training participants, train, predict the test participant."""
    allX = np.vstack([s[0][:, :nf] for s in train]); mu, sd = allX.mean(0), allX.std(0) + 1e-6
    norm = lambda s: (np.concatenate([(s[0][:, :nf] - mu) / sd, s[0][:, nf:]], 1).astype(np.float32), s[1], s[2])
    train, test = [norm(s) for s in train], [norm(s) for s in test]
    enc = (GRUEncoder(nf) if mtype == 'GRU' else GRUDEncoder(nf)).to(device); head = PredHead().to(device)
    y_all = np.concatenate([s[1][1:] if task == 'next' else s[1] for s in train])
    pw = torch.tensor([(y_all == 0).sum() / max((y_all == 1).sum(), 1)]).clamp(max=20).to(device)
    lf = nn.BCEWithLogitsLoss(pos_weight=pw); params = list(enc.parameters()) + list(head.parameters())
    opt = torch.optim.Adam(params, lr=1e-3, weight_decay=1e-4)
    dl = DataLoader([(s[0], s[1]) for s in train], batch_size=DEEP_BS, shuffle=True, collate_fn=collate)
    for _ in range(DEEP_EPOCHS):
        enc.train(); head.train()
        for X, Y, L in dl:
            X, Y = X.to(device), Y.to(device)
            H = enc(X, L) if mtype == 'GRU' else enc(X)
            pr, tg = (head(H[:, :-1]), Y[:, 1:]) if task == 'next' else (head(H), Y)
            mk = ~torch.isnan(tg)
            if mk.sum() == 0: continue
            loss = lf(pr[mk], tg[mk]); opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(params, 1.0); opt.step()
    enc.eval(); head.eval(); res = []
    with torch.no_grad():
        for X, y, pid in test:
            H = enc(torch.from_numpy(X).unsqueeze(0).to(device))
            if task == 'next':
                p = torch.sigmoid(head(H[:, :-1])).squeeze(0).cpu().numpy(); yy = y[1:]
            else:
                p = torch.sigmoid(head(H)).squeeze(0).cpu().numpy(); yy = y
            res.append((np.atleast_1d(p), yy, pid))
    return res
print("✅ models defined")


In [ ]:
# ════════════════════════════════════════════════════════
# CELL 4 — PARTICIPANT-LEVEL 10-FOLD CV (5 models x 2 feature sets x 2 tasks)
# ════════════════════════════════════════════════════════
MODELS = ['LR', 'RF', 'XGB', 'GRU', 'GRUD']

def run_config(mtype, fset):
    path = os.path.join(SAVE_DIR, 'predictions', f"{mtype}__{fset}.csv")
    if os.path.exists(path):
        print(f"  ✅ {mtype} {fset} (already done)"); return
    feats = FEATURE_SETS[fset]; nf = len(feats); t0 = time.time()
    seqs = build_seqs(feats, grud=(mtype == 'GRUD'))
    rows = []
    from sklearn.model_selection import KFold
    folds = list(KFold(n_splits=N_FOLDS if not QUICK_TEST else 2, shuffle=True, random_state=42).split(seqs))
    for k, (tr, te) in enumerate(folds):
        set_seed(SEED + k)
        train = [seqs[j] for j in tr]; test = [seqs[j] for j in te]
        for task in ['classify', 'next']:
            if mtype in ('LR', 'RF', 'XGB'):
                def flat(ss):
                    X = np.vstack([s[0][:-1] if task == 'next' else s[0] for s in ss])
                    y = np.concatenate([s[1][1:] if task == 'next' else s[1] for s in ss]); return X, y
                Xtr, ytr = flat(train); model = fit_flat(mtype, Xtr, ytr)
                preds = []
                for s in test:
                    Xte, yte = flat([s]); preds.append((model(Xte), yte, s[2]))
            else:
                preds = train_predict_deep(mtype, train, test, nf, task)
            for p, yy, pid in preds:
                rows += [{'task': task, 'id': pid, 'pos': i, 'y': int(a), 'p': float(b)}
                         for i, (a, b) in enumerate(zip(yy, p))]
    pd.DataFrame(rows).to_csv(path, index=False)
    print(f"  {mtype:4s} {fset:12s} done in {(time.time()-t0)/60:.1f} min")

for fset in FEATURE_SETS:
    for m in MODELS:
        run_config(m, fset)

# person-level baseline predictions (no model): running lapse rate before each EMA
bpath = os.path.join(SAVE_DIR, 'predictions', 'BASELINE__none.csv')
rows = []
for pid, gdf in df.groupby('id', sort=True):
    y = gdf['label'].to_numpy(int); s = gdf['baseline_score'].to_numpy()
    rows += [{'task': 'classify', 'id': pid, 'pos': i, 'y': y[i], 'p': s[i]} for i in range(len(y))]
    rows += [{'task': 'next', 'id': pid, 'pos': i, 'y': y[i + 1], 'p': s[i + 1]} for i in range(len(y) - 1)]
pd.DataFrame(rows).to_csv(bpath, index=False)
print("✅ all configurations done")


In [ ]:
# ════════════════════════════════════════════════════════
# CELL 5 — RESULTS: pooled AUC (cluster-bootstrap CI) and per-participant AUC
# ════════════════════════════════════════════════════════
rng = np.random.default_rng(SEED)
PRED = {}
for m, fset in [(m, f) for f in FEATURE_SETS for m in MODELS] + [('BASELINE', 'none')]:
    d = pd.read_csv(os.path.join(SAVE_DIR, 'predictions', f"{m}__{fset}.csv"), dtype={'id': str})
    for task in ['classify', 'next']:
        PRED[(m, fset, task)] = d[d.task == task].reset_index(drop=True)

IDS = sorted(df['id'].unique())
BOOT = [rng.choice(len(IDS), len(IDS), replace=True) for _ in range(N_BOOT)]   # shared by all configs

def by_id(d):
    return {pid: (g['y'].to_numpy(), g['p'].to_numpy()) for pid, g in d.groupby('id')}

def pooled(parts, idx):
    y = np.concatenate([parts[IDS[i]][0] for i in idx if IDS[i] in parts])
    p = np.concatenate([parts[IDS[i]][1] for i in idx if IDS[i] in parts])
    return roc_auc_score(y, p) if 0 < y.sum() < len(y) else np.nan

def per_person(parts):
    return pd.Series({pid: roc_auc_score(y, p) for pid, (y, p) in parts.items() if 0 < y.sum() < len(y)})

def boot_ci(fn):
    vals = np.array([fn(b) for b in BOOT]); vals = vals[~np.isnan(vals)]
    return np.percentile(vals, [2.5, 97.5]) if len(vals) else (np.nan, np.nan)

PARTS = {k: by_id(v) for k, v in PRED.items()}
rows = []
for (m, fset, task), parts in PARTS.items():
    pp = per_person(parts); full = list(range(len(IDS)))
    lo, hi = boot_ci(lambda b: pooled(parts, b))
    gap_lo, gap_hi = boot_ci(lambda b: pooled(parts, b) - np.nanmedian(
        [pp.get(IDS[i], np.nan) for i in b]))
    rows.append({'model': m, 'features': fset, 'task': task,
                 'pooled_AUC': round(pooled(parts, full), 3), 'pooled_CI_low': round(lo, 3), 'pooled_CI_high': round(hi, 3),
                 'per_participant_median': round(pp.median(), 3),
                 'pp_IQR_low': round(pp.quantile(.25), 3), 'pp_IQR_high': round(pp.quantile(.75), 3), 'n_pp': len(pp),
                 'pooled_minus_median': round(pooled(parts, full) - pp.median(), 3),
                 'gap_CI_low': round(gap_lo, 3), 'gap_CI_high': round(gap_hi, 3)})
results = pd.DataFrame(rows)
results.to_csv(os.path.join(SAVE_DIR, 'results_table.csv'), index=False)
print(results.to_string(index=False))


In [ ]:
# ════════════════════════════════════════════════════════
# CELL 6 — THE THREE PRE-SPECIFIED CHECKS
# Paired comparisons use the same bootstrap resamples (pooled) and the same
# participants (within-person Wilcoxon). Directional evidence: report the
# direction and interval; Holm-adjusted p is shown within each check for transparency.
# ════════════════════════════════════════════════════════
def holm(p):
    p = np.asarray(p, float); adj = np.empty(len(p)); run = 0.0
    for r, i in enumerate(np.argsort(p)):
        run = max(run, (len(p) - r) * p[i]); adj[i] = min(1.0, run)
    return adj

def compare(a, b, task):
    """A minus B: pooled difference with paired bootstrap CI, and within-person paired test."""
    pa, pb = PARTS[a + (task,)], PARTS[b + (task,)]; full = list(range(len(IDS)))
    d_pool = pooled(pa, full) - pooled(pb, full)
    lo, hi = boot_ci(lambda bi: pooled(pa, bi) - pooled(pb, bi))
    sa, sb = per_person(pa), per_person(pb); common = sa.index.intersection(sb.index)
    d = (sa[common] - sb[common]).to_numpy()
    w = stats.wilcoxon(d).pvalue if np.any(d != 0) else 1.0
    return {'task': task, 'A': ' '.join(a), 'B': ' '.join(b),
            'pooled_diff': round(d_pool, 3), 'pooled_diff_CI_low': round(lo, 3), 'pooled_diff_CI_high': round(hi, 3),
            'within_median_diff': round(float(np.median(d)), 3),
            'share_A_better': round(float((d > 0).mean()), 2), 'n_participants': len(d),
            'wilcoxon_p': w}

def summarise(tab, name):
    tab['wilcoxon_p_holm'] = tab.groupby('task')['wilcoxon_p'].transform(holm)
    tab.to_csv(os.path.join(SAVE_DIR, f"{name}.csv"), index=False)
    print(f"\n{name}\n" + tab.round(3).to_string(index=False))

# Check 1 — pooled vs per-participant AUC (from Cell 5)
chk1 = results[['model', 'features', 'task', 'pooled_AUC', 'per_participant_median',
                'pooled_minus_median', 'gap_CI_low', 'gap_CI_high', 'n_pp']]
chk1.to_csv(os.path.join(SAVE_DIR, 'check1_pooled_vs_per_participant.csv'), index=False)
print("Check 1 — pooled AUC minus per-participant median (CI from participant bootstrap)")
print(chk1.to_string(index=False))
print(f"Configurations where pooled > per-participant median: "
      f"{(chk1.pooled_minus_median > 0).sum()} of {len(chk1)}; "
      f"lower CI bound above 0: {(chk1.gap_CI_low > 0).sum()}")

# Check 2 — expanded vs original, per model and feature family
summarise(pd.DataFrame([compare((m, f'{fam}_exp'), (m, f'{fam}_orig'), t)
                        for t in ['classify', 'next'] for fam in ['native'] for m in MODELS]),
          'check2_expanded_vs_original')

# Check 3 — GRU-D vs GRU, per feature set
summarise(pd.DataFrame([compare(('GRUD', f), ('GRU', f), t)
                        for t in ['classify', 'next'] for f in FEATURE_SETS]),
          'check3_grud_vs_gru')
print("\nEMASENS (regular timing), pooled GRU-D minus GRU: classification +0.040 to +0.058; next-entry -0.001 to +0.029")
print("Smoke Free, for comparison (rerun 1 Oct 2026, pooled GRU-D minus GRU): "
      "original +0.005 (classification) / +0.010 (next-entry); expanded +0.013 / +0.015")

# Check 4 — how far does the person-level baseline get?
summarise(pd.DataFrame([compare((m, f), ('BASELINE', 'none'), t)
                        for t in ['classify', 'next'] for f in FEATURE_SETS for m in MODELS]),
          'check4_models_vs_person_baseline')
b = results[results.model == 'BASELINE']
print("\nPerson-level baseline alone:"); print(b[['task', 'pooled_AUC', 'pooled_CI_low', 'pooled_CI_high',
                                                 'per_participant_median', 'n_pp']].to_string(index=False))


## References

- Wyant, K., Sant'Ana, S. J., Fronk, G. E., & Curtin, J. J. (2024). Machine learning models for temporally
  precise lapse prediction in alcohol use disorder. *Journal of Psychopathology and Clinical Science, 133*(7),
  527–540. https://doi.org/10.1037/abn0000901
- Wyant, K., Sant'Ana, S. J. K., Fronk, G., & Curtin, J. J. (2023). *Machine learning models for temporally
  precise lapse prediction in alcohol use disorder* [OSF project: data, scripts, supplement].
  https://osf.io/w5h9y (no licence attached).
- Che, Z., Purushotham, S., Cho, K., Sontag, D., & Liu, Y. (2018). Recurrent neural networks for
  multivariate time series with missing values. *Scientific Reports, 8*, 6085.
